In [1]:
from pyspark.sql import SparkSession

spark = SparkSession\
        .builder\
        .appName("Cours de Spark")\
        .master("local[*]")\
        .getOrCreate()
sc = spark.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
df_used = spark.read.options(header=True, inferSchema=True).csv("gps_user.csv")

In [3]:
count_original = df_used.count()
count_distinct = df_used.distinct().count()
count_duplicates = count_original - count_distinct
print(count_original, count_distinct, count_duplicates)

64295 30678 33617


In [4]:
df_unique = df_used.dropDuplicates(['App'])
count_app_distinct = df_unique.count()
print(count_original, count_distinct, count_app_distinct)

64295 30678 1074


In [8]:
import numpy as np
from pyspark.sql.functions import isnan, isnull, col
from pyspark.sql.types import BooleanType

def getMissingValues(dataframe):
  count = dataframe.count()
  columns = dataframe.columns
  nan_count = []
  # we can't check for nan in a boolean type column
  for column in columns:
    if dataframe.schema[column].dataType == BooleanType():
      nan_count.append(0)
    else:
      nan_count.append(dataframe.where(isnan(col(column))).count())
  null_count = [dataframe.where(isnull(col(column))).count() for column in columns]
  return([count, columns, nan_count, null_count])

def missingTable(stats):
  count, columns, nan_count, null_count = stats
  count = str(count)
  nan_count = [str(element) for element in nan_count]
  null_count = [str(element) for element in null_count]
  max_init = np.max([len(str(count)), 10])
  line1 = "+" + max_init*"-" + "+"
  line2 = "|" + (max_init-len(count))*" " + count + "|"
  line3 = "|" + (max_init-9)*" " + "nan count|"
  line4 = "|" + (max_init-10)*" " + "null count|"
  for i in range(len(columns)):
    max_column = np.max([len(columns[i]),\
                        len(nan_count[i]),\
                        len(null_count[i])])
    line1 += max_column*"-" + "+"
    line2 += (max_column - len(columns[i]))*" " + columns[i] + "|"
    line3 += (max_column - len(nan_count[i]))*" " + nan_count[i] + "|"
    line4 += (max_column - len(null_count[i]))*" " + null_count[i] + "|"
  lines = f"{line1}\n{line2}\n{line1}\n{line3}\n{line4}\n{line1}"
  print(lines)

missingTable(getMissingValues(df_unique))

+----------+---+-----------------+---------+------------------+----------------------+
|      1074|App|Translated_Review|Sentiment|Sentiment_Polarity|Sentiment_Subjectivity|
+----------+---+-----------------+---------+------------------+----------------------+
| nan count|  0|              433|      433|               433|                   433|
|null count|  0|                0|        0|                 0|                     0|
+----------+---+-----------------+---------+------------------+----------------------+


In [9]:
df_joined = df_final.join(df_user, "App")
df_joined.filter(~isnan("Sentiment_Polarity") & ~isnull("Sentiment_Polarity"))\
         .select("App", col("Sentiment_Polarity").cast("double"))\
         .groupBy("App")\
         .avg("Sentiment_Polarity")\
         .sort("avg(Sentiment_Polarity)", ascending=False)\
         .show()

NameError: name 'df_final' is not defined